In [ ]:

import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
reviews_df = session.sql("SELECT * FROM ANTAM.DATA.TRANSACTION_DATA ").to_pandas()
reviews_df

In [ ]:
pd.set_option('display.max_columns', None)
print(df_antam.head(2))

In [ ]:
kolom_penting = ['INVOICEDATE', 'LOCATION', 'NAMABARANG', 'QTYOUT']
# Pastikan hanya kolom yang tersedia diambil (antisipasi nama beda)
kolom_tersedia = [kol for kol in kolom_penting if kol in df_antam.columns]
df_forcast = df_antam[kolom_tersedia].copy()

In [ ]:
# Ubah QTYOUT ke numeric, isi NaN jadi 0
df_forcast['QTYOUT'] = pd.to_numeric(df_forcast['QTYOUT'], errors='coerce').fillna(0)

In [ ]:
# Hapus baris yang tanggal atau nama barang/lokasi-nya kosong
df_forcast = df_forcast.dropna(subset=['INVOICEDATE', 'LOCATION', 'NAMABARANG'])


In [ ]:
print(df_forcast.head())

In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# from xgboost import XGBRegressor
# from sklearn.metrics import mean_squared_error

# # === 1️⃣ Gunakan data yang sudah ada ===
# df = df_forcast.copy()

# # Pastikan tipe tanggal benar
# df['INVOICEDATE'] = pd.to_datetime(df['INVOICEDATE'])

# # === 2️⃣ Fitur waktu tambahan ===
# df['dayofweek'] = df['INVOICEDATE'].dt.dayofweek
# df['month'] = df['INVOICEDATE'].dt.month

# # === 3️⃣ Tambahkan fitur lag untuk pola historis ===
# df = df.sort_values(['LOCATION', 'NAMABARANG', 'INVOICEDATE'])
# df['lag_7'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(7)
# df['lag_30'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(30)
# df['lag_90'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(90)

# # Isi NaN pada awal data dengan median per group
# df[['lag_7', 'lag_30', 'lag_90']] = df[['lag_7', 'lag_30', 'lag_90']].fillna(method='bfill')

# # === 4️⃣ Loop setiap lokasi & barang untuk model masing-masing ===
# results = []

# for (loc, barang), group in df.groupby(['LOCATION', 'NAMABARANG']):
#     group = group.sort_values('INVOICEDATE').copy()

#     # Gunakan fitur
#     X = group[['dayofweek', 'month', 'lag_7', 'lag_30', 'lag_90']]
#     y = group['QTYOUT']

#     # === 5️⃣ Split train dan test (80% train, 20% test) ===
#     split_idx = int(len(group) * 0.8)
#     X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
#     y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

#     # === 6️⃣ Train model ===
#     model = XGBRegressor(
#         n_estimators=200,
#         learning_rate=0.1,
#         max_depth=5,
#         random_state=42
#     )
#     model.fit(X_train, y_train)

#     # === 7️⃣ Prediksi test ===
#     y_pred_test = model.predict(X_test)

#     # === 8️⃣ Forecast 90 hari ke depan ===
#     last_date = group['INVOICEDATE'].max()
#     future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=90)

#     future_df = pd.DataFrame({
#         'INVOICEDATE': future_dates,
#         'dayofweek': future_dates.dayofweek,
#         'month': future_dates.month,
#     })

#     # Inisialisasi lag dengan data terakhir
#     last_lag7 = y.iloc[-7:].mean()
#     last_lag30 = y.iloc[-30:].mean()
#     last_lag90 = y.iloc[-90:].mean()

#     preds = []
#     for date in future_dates:
#         X_future = np.array([[date.dayofweek, date.month, last_lag7, last_lag30, last_lag90]])
#         pred = model.predict(X_future)[0]
#         preds.append(pred)
#         # Update lag agar dinamis
#         last_lag7 = (last_lag7 * 6 + pred) / 7
#         last_lag30 = (last_lag30 * 29 + pred) / 30
#         last_lag90 = (last_lag90 * 89 + pred) / 90

#     future_df['forecast'] = preds

#     # === 9️⃣ Simpan hasil ===
#     results.append({
#         'LOCATION': loc,
#         'NAMABARANG': barang,
#         'train_pred': y_train,
#         'test_actual': y_test,
#         'test_pred': y_pred_test,
#         'forecast_df': future_df,
#         'rmse': np.sqrt(mean_squared_error(y_test, y_pred_test))
#     })

#     # === 🔟 Plot hasil ===
#     plt.figure(figsize=(10,5))
#     plt.plot(group['INVOICEDATE'][:split_idx], y_train, label='Train', color='blue')
#     plt.plot(group['INVOICEDATE'][split_idx:], y_test, label='Test', color='orange')
#     plt.plot(group['INVOICEDATE'][split_idx:], y_pred_test, label='Predicted (Test)', color='green', linestyle='--')
#     plt.plot(future_df['INVOICEDATE'], future_df['forecast'], label='Forecast 90 Days', color='red')
#     plt.title(f'Forecasting 90 Hari - {barang} @ {loc}\nRMSE: {np.sqrt(mean_squared_error(y_test, y_pred_test)):.2f}')
#     plt.xlabel('Tanggal')
#     plt.ylabel('QTYOUT')
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

# # === 11️⃣ Rekap RMSE tiap model ===
# summary = pd.DataFrame(results)[['LOCATION', 'NAMABARANG', 'rmse']]
# print("\n=== Evaluasi Model per Lokasi & Barang ===")
# print(summary)


In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# from xgboost import XGBRegressor
# from sklearn.metrics import mean_squared_error

# # === 1️⃣ Gunakan Data yang Sudah Ada ===
# df = df_forcast.copy()

# # Pastikan kolom tanggal benar
# df['INVOICEDATE'] = pd.to_datetime(df['INVOICEDATE'])

# # === 2️⃣ Tambahkan fitur waktu dan lag ===
# df = df.sort_values(['LOCATION', 'NAMABARANG', 'INVOICEDATE'])
# df['dayofweek'] = df['INVOICEDATE'].dt.dayofweek
# df['month'] = df['INVOICEDATE'].dt.month
# df['lag_7'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(7)
# df['lag_30'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(30)
# df['lag_90'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(90)
# df[['lag_7', 'lag_30', 'lag_90']] = df[['lag_7', 'lag_30', 'lag_90']].fillna(method='bfill')

# # === 3️⃣ Simpan hasil untuk rekap ===
# results = []

# # === 4️⃣ Loop per lokasi dan barang ===
# for (loc, barang), group in df.groupby(['LOCATION', 'NAMABARANG']):
#     group = group.sort_values('INVOICEDATE').copy()

#     # Siapkan fitur dan target
#     X = group[['dayofweek', 'month', 'lag_7', 'lag_30', 'lag_90']]
#     y = group['QTYOUT']

#     # Split train/test (80% train, 20% test)
#     split_idx = int(len(group) * 0.8)
#     X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
#     y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

#     # Train model
#     model = XGBRegressor(
#         n_estimators=200,
#         learning_rate=0.1,
#         max_depth=5,
#         random_state=42
#     )
#     model.fit(X_train, y_train)

#     # Prediksi train dan test
#     y_pred_train = model.predict(X_train)
#     y_pred_test = model.predict(X_test)

#     # === 5️⃣ Forecast 90 hari ke depan ===
#     last_date = group['INVOICEDATE'].max()
#     future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=90)
#     future_df = pd.DataFrame({
#         'INVOICEDATE': future_dates,
#         'dayofweek': future_dates.dayofweek,
#         'month': future_dates.month
#     })

#     # Gunakan lag terakhir dari data historis
#     last_lag7 = y.iloc[-7:].mean()
#     last_lag30 = y.iloc[-30:].mean()
#     last_lag90 = y.iloc[-90:].mean()

#     preds = []
#     for date in future_dates:
#         X_future = np.array([[date.dayofweek, date.month, last_lag7, last_lag30, last_lag90]])
#         pred = model.predict(X_future)[0]
#         preds.append(pred)
#         # update lag
#         last_lag7 = (last_lag7 * 6 + pred) / 7
#         last_lag30 = (last_lag30 * 29 + pred) / 30
#         last_lag90 = (last_lag90 * 89 + pred) / 90

#     future_df['forecast'] = preds

#     # === 6️⃣ Evaluasi ===
#     rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
#     rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

#     # Simpan hasil
#     results.append({
#         'LOCATION': loc,
#         'NAMABARANG': barang,
#         'rmse_train': rmse_train,
#         'rmse_test': rmse_test,
#         'forecast_df': future_df
#     })

#     # === 7️⃣ Plot Train/Test/Forecast + Data Aktual ===
#     plt.figure(figsize=(12,6))
#     plt.plot(group['INVOICEDATE'], y, label='Data Aktual', color='black', linewidth=2)
#     plt.plot(group['INVOICEDATE'][:split_idx], y_pred_train, label='Prediksi Train', color='blue', linestyle='--')
#     plt.plot(group['INVOICEDATE'][split_idx:], y_pred_test, label='Prediksi Test', color='green', linestyle='--')
#     plt.plot(future_df['INVOICEDATE'], future_df['forecast'], label='Forecast 90 Hari', color='red', linewidth=2)

#     plt.title(f"{barang} @ {loc}\nRMSE Train={rmse_train:.2f}, RMSE Test={rmse_test:.2f}")
#     plt.xlabel("Tanggal")
#     plt.ylabel("QTYOUT")
#     plt.legend()
#     plt.grid(True, linestyle='--', alpha=0.5)
#     plt.tight_layout()
#     plt.show()

# # === 8️⃣ Ringkasan performa model ===
# summary = pd.DataFrame(results)[['LOCATION', 'NAMABARANG', 'rmse_train', 'rmse_test']]
# print("\n=== Evaluasi Model per Lokasi & Barang ===")
# print(summary)




In [ ]:
# import pandas as pd
# import numpy as np
# import matplotlib.pyplot as plt
# from xgboost import XGBRegressor
# from sklearn.metrics import mean_squared_error

# # === 1️⃣ Gunakan Data yang Sudah Ada ===
# df = df_forcast.copy()

# # Pastikan kolom tanggal benar
# df['INVOICEDATE'] = pd.to_datetime(df['INVOICEDATE'])

# # === 2️⃣ Tambahkan fitur waktu dan lag ===
# df = df.sort_values(['LOCATION', 'NAMABARANG', 'INVOICEDATE'])
# df['dayofweek'] = df['INVOICEDATE'].dt.dayofweek
# df['month'] = df['INVOICEDATE'].dt.month
# df['lag_7'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(7)
# df['lag_30'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(30)
# df['lag_90'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(90)
# df[['lag_7', 'lag_30', 'lag_90']] = df[['lag_7', 'lag_30', 'lag_90']].fillna(method='bfill')

# # === 3️⃣ Simpan hasil untuk rekap ===
# results = []
# final_frames = []  # untuk gabung semua data (aktual + train/test + forecast)

# # === 4️⃣ Loop per lokasi dan barang ===
# for (loc, barang), group in df.groupby(['LOCATION', 'NAMABARANG']):
#     group = group.sort_values('INVOICEDATE').copy()

#     # Siapkan fitur dan target
#     X = group[['dayofweek', 'month', 'lag_7', 'lag_30', 'lag_90']]
#     y = group['QTYOUT']

#     # Split train/test (80% train, 20% test)
#     split_idx = int(len(group) * 0.8)
#     X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
#     y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

#     # Train model
#     model = XGBRegressor(
#         n_estimators=200,
#         learning_rate=0.1,
#         max_depth=5,
#         random_state=42
#     )
#     model.fit(X_train, y_train)

#     # Prediksi train dan test
#     y_pred_train = model.predict(X_train)
#     y_pred_test = model.predict(X_test)

#     # === 5️⃣ Forecast 90 hari ke depan ===
#     last_date = group['INVOICEDATE'].max()
#     future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=90)
#     future_df = pd.DataFrame({
#         'INVOICEDATE': future_dates,
#         'dayofweek': future_dates.dayofweek,
#         'month': future_dates.month
#     })

#     # Gunakan lag terakhir dari data historis
#     last_lag7 = y.iloc[-7:].mean()
#     last_lag30 = y.iloc[-30:].mean()
#     last_lag90 = y.iloc[-90:].mean()

#     preds = []
#     for date in future_dates:
#         X_future = np.array([[date.dayofweek, date.month, last_lag7, last_lag30, last_lag90]])
#         pred = model.predict(X_future)[0]
#         preds.append(pred)
#         # update lag
#         last_lag7 = (last_lag7 * 6 + pred) / 7
#         last_lag30 = (last_lag30 * 29 + pred) / 30
#         last_lag90 = (last_lag90 * 89 + pred) / 90

#     future_df['FORECAST_PRED'] = preds
#     future_df['LOCATION'] = loc
#     future_df['NAMABARANG'] = barang
#     future_df['ACTUAL'] = np.nan
#     future_df['TRAIN_PRED'] = np.nan
#     future_df['TEST_PRED'] = np.nan

#     # === 6️⃣ Gabungkan aktual + prediksi train/test ===
#     df_actual = group[['INVOICEDATE', 'QTYOUT']].copy()
#     df_actual['LOCATION'] = loc
#     df_actual['NAMABARANG'] = barang
#     df_actual['ACTUAL'] = df_actual['QTYOUT']
#     df_actual['TRAIN_PRED'] = np.nan
#     df_actual['TEST_PRED'] = np.nan
#     df_actual['FORECAST_PRED'] = np.nan

#     df_actual.loc[df_actual.index[:split_idx], 'TRAIN_PRED'] = y_pred_train
#     df_actual.loc[df_actual.index[split_idx:], 'TEST_PRED'] = y_pred_test

#     df_combined = pd.concat([df_actual.drop(columns=['QTYOUT']), future_df], ignore_index=True)
#     final_frames.append(df_combined)

#     # === 7️⃣ Evaluasi ===
#     rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
#     rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))

#     results.append({
#         'LOCATION': loc,
#         'NAMABARANG': barang,
#         'rmse_train': rmse_train,
#         'rmse_test': rmse_test
#     })

#     # === 8️⃣ Plot Grafik ===
#     plt.figure(figsize=(12,6))
#     plt.plot(group['INVOICEDATE'], y, label='Data Aktual', color='black', linewidth=2)
#     plt.plot(group['INVOICEDATE'][:split_idx], y_pred_train, label='Prediksi Train', color='blue', linestyle='--')
#     plt.plot(group['INVOICEDATE'][split_idx:], y_pred_test, label='Prediksi Test', color='green', linestyle='--')
#     plt.plot(future_df['INVOICEDATE'], future_df['FORECAST_PRED'], label='Forecast 90 Hari', color='red', linewidth=2)

#     plt.title(f"{barang} @ {loc}\nRMSE Train={rmse_train:.2f}, RMSE Test={rmse_test:.2f}")
#     plt.xlabel("Tanggal")
#     plt.ylabel("QTYOUT")
#     plt.legend()
#     plt.grid(True, linestyle='--', alpha=0.5)
#     plt.tight_layout()
#     plt.show()

# # === 9️⃣ Ringkasan performa model ===
# summary = pd.DataFrame(results)
# print("\n=== Evaluasi Model per Lokasi & Barang ===")
# print(summary)

# # === 🔟 Gabungkan semua hasil ke satu DataFrame ===
# df_result_all = pd.concat(final_frames, ignore_index=True)
# df_result_all = df_result_all.sort_values(['LOCATION', 'NAMABARANG', 'INVOICEDATE'])

# print(df_result_all.head(10))


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
from snowflake.ml.registry import Registry
from datetime import datetime
import re
import warnings

# --- IMPORT TAMBAHAN UNTUK REGISTRY DETAIL ---
# Mengimpor modul task dan type_hints dari snowflake.ml.model
from snowflake.ml.model import task, type_hints

# Mengabaikan warning yang mungkin muncul selama iterasi
warnings.filterwarnings('ignore', category=FutureWarning)

# ============================================================
# 🧩 HELPER FUNCTION UNTUK LOGGING
# ============================================================
def log(msg):
    """Mencetak pesan log dengan timestamp."""
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

# ============================================================
# 🧠 GUNAKAN SESSION DEFAULT DARI NOTEBOOK SNOWFLAKE
# ASUMSI: variabel 'session' sudah tersedia di lingkungan notebook.
# ============================================================
# Pastikan session sudah didefinisikan di lingkungan Anda
try:
    sp_session = session
except NameError:
    log("⚠️ Variabel 'session' (Snowpark Session) tidak ditemukan.")
    raise

# ASUMSI: variabel 'df_forcast' sudah tersedia
try:
    df = df_forcast.copy()
except NameError:
    log("⚠️ Variabel 'df_forcast' (DataFrame data) tidak ditemukan.")
    raise

# ============================================================
# 🧱 INISIALISASI REGISTRY
# ============================================================
reg = Registry(session=sp_session, database_name="ANTAM", schema_name="REGISTRY")

# ============================================================
# 📦 SIAPKAN DATA
# ============================================================
df['INVOICEDATE'] = pd.to_datetime(df['INVOICEDATE'])
df = df.sort_values(['LOCATION', 'NAMABARANG', 'INVOICEDATE'])

# Tambahkan fitur waktu & lag
df['dayofweek'] = df['INVOICEDATE'].dt.dayofweek
df['month'] = df['INVOICEDATE'].dt.month
# Menggunakan 'bfill' agar data awal tidak dihilangkan saat 'dropna'
df['lag_7']  = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(7)
df['lag_30'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(30)
df['lag_90'] = df.groupby(['LOCATION', 'NAMABARANG'])['QTYOUT'].shift(90)
df[['lag_7', 'lag_30', 'lag_90']] = df[['lag_7', 'lag_30', 'lag_90']].bfill() # Mengisi NaN dengan nilai selanjutnya

# Menghilangkan baris dengan NaN (jika masih ada)
df.dropna(subset=['lag_7', 'lag_30', 'lag_90'], inplace=True)

log(f"Data prepared: {len(df):,} rows")

results = []
final_frames = []

# ============================================================
# 🔁 LOOP PER (LOKASI, NAMABARANG)
# ============================================================
for (loc, barang), group in df.groupby(['LOCATION', 'NAMABARANG']):
    try:
        start = datetime.now()
        log(f"Start: {barang} @ {loc} — {len(group)} rows")

        if len(group) < 40:
            log(f"  ⚠️ Skip: data terlalu sedikit ({len(group)} rows)")
            continue

        group = group.sort_values('INVOICEDATE').reset_index(drop=True)
        # Kolom fitur
        feature_cols = ['dayofweek', 'month', 'lag_7', 'lag_30', 'lag_90']
        X = group[feature_cols]
        y = group['QTYOUT']

        split_idx = int(len(group) * 0.8)
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

        # ============================================================
        # 🏋️ TRAINING MODEL
        # ============================================================
        model = XGBRegressor(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=5,
            random_state=42,
            n_jobs=-1 # Gunakan semua core yang tersedia
        )
        model.fit(X_train, y_train)
        log("  ✅ Training completed.")
        
        # ============================================================
        # 📈 EVALUASI MODEL (Diperlukan sebelum logging untuk mendapatkan metrics)
        # ============================================================
        y_pred_train = model.predict(X_train)
        y_pred_test = model.predict(X_test)

        rmse_train = float(np.sqrt(mean_squared_error(y_train, y_pred_train)))
        rmse_test = float(np.sqrt(mean_squared_error(y_test, y_pred_test)))
        
        # Siapkan dictionary metrics
        model_metrics = {
            'rmse_train': rmse_train,
            'rmse_test': rmse_test,
        }

        # ============================================================
        # 💾 SIMPAN MODEL KE REGISTRY (Sintaks Eksplisit)
        # ============================================================
        raw_name = f"XGB_FORECAST_{loc}_{barang}"
        safe_name = re.sub(r'[^A-Za-z0-9_]+', '_', raw_name).upper()
        
        # Ambil satu baris data sebagai sample_input_data 
        sample_input_data = X_train.iloc[[0]].copy()

        try:
            mv = reg.log_model(
                model=model,
                model_name=safe_name,
                # FIX 1: Hapus version_name="v1" agar Registry otomatis membuat V2, V3, dst.
                conda_dependencies=["xgboost", "pandas", "scikit-learn"], # FIX 2: Tambahkan 'scikit-learn'
                comment=f"Model forecasting untuk {barang} di lokasi {loc}", # Komentar
                metrics=model_metrics,  # Metrik RMSE
                sample_input_data=sample_input_data,  # Contoh input
                task=task.Task.TABULAR_REGRESSION  # Tipe tugas adalah Regresi
            )
            
            # Ambil informasi model version
            mv_name = getattr(mv, "model_name", safe_name)
            # FIX 3: Tetap gunakan version_name untuk logging
            mv_ver = getattr(mv, "version_name", "N/A") 
            log(f"  ✅ Model tersimpan di registry: {mv_name} (version: {mv_ver})")
            
        except Exception as e:
            # Tingkatkan pesan error jika ada masalah di Registry atau Permissions
            error_msg = str(e).split('\n')[0] 
            log(f"  ❌ Gagal menyimpan model {safe_name}: {error_msg}. Cek permissions dan keberadaan DB/Schema.")

        # ============================================================
        # 🔮 PREDIKSI DAN FORECAST 90 HARI
        # ============================================================
        # Prediksi y_pred_train dan y_pred_test sudah dihitung di bagian evaluasi
        
        last_date = group['INVOICEDATE'].max()
        future_dates = pd.date_range(last_date + pd.Timedelta(days=1), periods=90)

        # Inisialisasi lag untuk forecasting dari data aktual terakhir
        last_lag7  = group['QTYOUT'].iloc[-7:].mean() if len(group) >= 7 else group['QTYOUT'].mean()
        last_lag30 = group['QTYOUT'].iloc[-30:].mean() if len(group) >= 30 else group['QTYOUT'].mean()
        last_lag90 = group['QTYOUT'].iloc[-90:].mean() if len(group) >= 90 else group['QTYOUT'].mean()

        preds = []
        for d in future_dates:
            # Gunakan nilai lag yang diperbarui
            X_future = pd.DataFrame({
                'dayofweek': [d.dayofweek], 
                'month': [d.month], 
                'lag_7': [last_lag7], 
                'lag_30': [last_lag30], 
                'lag_90': [last_lag90]
            })
            # Pastikan nama kolom sesuai
            X_future = X_future[feature_cols]

            pred = float(model.predict(X_future)[0])
            preds.append(pred)
            
            # Update lag secara dinamis (menggunakan Exponential Moving Average sederhana)
            last_lag7  = (last_lag7 * 6 + pred) / 7
            last_lag30 = (last_lag30 * 29 + pred) / 30
            last_lag90 = (last_lag90 * 89 + pred) / 90

        future_df = pd.DataFrame({
            'INVOICEDATE': future_dates,
            'LOCATION': loc,
            'NAMABARANG': barang,
            'ACTUAL': np.nan,
            'TRAIN_PRED': np.nan,
            'TEST_PRED': np.nan,
            'FORECAST_PRED': preds
        })

        # ============================================================
        # 📊 GABUNGKAN AKTUAL + TRAIN + TEST + FORECAST
        # ============================================================
        df_actual = group[['INVOICEDATE', 'QTYOUT']].copy()
        df_actual['LOCATION'] = loc
        df_actual['NAMABARANG'] = barang
        df_actual.rename(columns={'QTYOUT': 'ACTUAL'}, inplace=True)
        df_actual['TRAIN_PRED'] = np.nan
        df_actual['TEST_PRED'] = np.nan
        df_actual['FORECAST_PRED'] = np.nan
        df_actual.loc[df_actual.index[:split_idx], 'TRAIN_PRED'] = y_pred_train
        df_actual.loc[df_actual.index[split_idx:], 'TEST_PRED'] = y_pred_test

        combined = pd.concat([df_actual, future_df], ignore_index=True)
        final_frames.append(combined)

        # ============================================================
        # 📈 EVALUASI MODEL (Metrik sudah dihitung di atas)
        # ============================================================
        results.append({
            'LOCATION': loc,
            'NAMABARANG': barang,
            'rmse_train': rmse_train,
            'rmse_test': rmse_test,
            'model_name': safe_name
        })

        # ============================================================
        # 🖼️ PLOT HASIL
        # ============================================================
        plt.figure(figsize=(12,6))
        # Plot data Aktual (Historical)
        plt.plot(group['INVOICEDATE'], y, label='Aktual', color='#1f77b4', linewidth=2)
        
        # Plot prediksi Train dan Test
        plt.plot(group['INVOICEDATE'][:split_idx], y_pred_train, label='Prediksi Train', color='#ff7f0e', linestyle='--')
        plt.plot(group['INVOICEDATE'][split_idx:], y_pred_test, label='Prediksi Test', color='#2ca02c', linestyle='--')
        
        # Plot Forecast
        plt.plot(future_df['INVOICEDATE'], future_df['FORECAST_PRED'], label='Forecast 90 Hari', color='#d62728', linewidth=2, marker='o', markersize=3)
        
        plt.axvline(x=group['INVOICEDATE'].iloc[split_idx], color='gray', linestyle=':', label='Batas Train/Test')
        
        plt.title(f"Forecasting {barang} @ {loc} — RMSE Train={rmse_train:.2f}, Test={rmse_test:.2f}", fontsize=14)
        plt.xlabel("Tanggal", fontsize=12)
        plt.ylabel("QTYOUT", fontsize=12)
        plt.legend()
        plt.grid(alpha=0.4)
        plt.tight_layout()
        plt.show()

        elapsed = (datetime.now() - start).total_seconds()
        log(f"  ✅ Selesai: {barang} @ {loc} ({elapsed:.1f}s) — RMSE Train={rmse_train:.2f}, Test={rmse_test:.2f}")

    except Exception as e:
        log(f"❌ Error di {barang} @ {loc}: {e}")
        continue

# ============================================================
# 📜 RINGKASAN HASIL
# ============================================================
summary = pd.DataFrame(results)
log("\n=== Ringkasan Model ===")
print(summary)

if final_frames:
    df_result_all = pd.concat(final_frames, ignore_index=True)
    df_result_all = df_result_all.sort_values(['LOCATION', 'NAMABARANG', 'INVOICEDATE']).reset_index(drop=True)
    log(f"Total hasil: {len(df_result_all):,} baris")
    print("\n--- 10 Baris Hasil Keseluruhan ---")
    print(df_result_all.head(10))

    # ============================================================
    # 💾 SIMPAN HASIL KE SNOWFLAKE TABLE
    # ============================================================
    try:
        sp_session.write_pandas(summary, "FORECAST_MODEL_SUMMARY", auto_create_table=True, overwrite=True)
        sp_session.write_pandas(df_result_all, "FORECAST_RESULTS_ALL", auto_create_table=True, overwrite=True)
        log("✅ Data tersimpan ke tabel FORECAST_MODEL_SUMMARY dan FORECAST_RESULTS_ALL")
    except Exception as e:
        log(f"⚠️ Gagal menyimpan hasil ke tabel: {e}")


In [ ]:
df_result_all


In [ ]:
from snowflake.snowpark.types import StructType, StructField, DateType, StringType, FloatType

schema = StructType([
    StructField("INVOICEDATE", DateType()),
    StructField("LOCATION", StringType()),
    StructField("NAMABARANG", StringType()),
    StructField("FORECAST_PRED", FloatType())
])

df_sp = sp_session.create_dataframe(df_result_all, schema=schema)
df_sp.write.save_as_table("FORECAST_RESULT_ALL", mode="overwrite")


In [ ]:
df = session.sql("SELECT * FROM ANTAM.DATA.FORECAST_RESULT_ALL ").to_pandas()
df